# 08 — ETL Geografía + Renta a nivel barri (opendata_barcelona)

Incorpora datos de `data/raw/opendata_barcelona/` que la fase ETL 01-07 había dejado fuera
y que SÍ sirven a los objetivos del proyecto:

1. **`contexto_renta_barri.csv`** — renta a nivel BARRI (de la renta por sección censal,
   2015-2021). Cruzable con `fact_incidentes_gub` por `id_territori` → **economía vs delincuencia**.
2. **`geo_territorio.csv`** — geometrías WGS84 (lat/lon) de barris y ABPs → **ML 'predecir DÓNDE'**
   y enlace espacial barri↔ABP. Se une a `dim_territorio` por `id_territorio`.

**Por qué no se modifica dim_territorio:** las geometrías van en tabla aparte para no alterar
los ids ya cargados por los facts; se unen por `id_territorio`.

**Esquemas:**
- contexto_renta_barri: `anyo, id_territori, territori, nivel_territorial, indicador, categoria, sexe, valor, unitat` (igual que contexto_socioeconomico → se puede concatenar)
- geo_territorio: `id_territorio, nivel_territorial, codi, nom, geometria_wgs84`

El match con `dim_territorio` se hace por **código** (robusto), no por nombre.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

BASE = Path(r'E:\informacion y documentos\Curso analisis de datos IT Academy\Proyecto final\criminalistica_cat')
RAW  = BASE / 'data' / 'raw'
CLEAN = BASE / 'data' / 'clean'
OD = RAW / 'opendata_barcelona'

dim_territorio = pd.read_csv(CLEAN / 'dim_territorio.csv', encoding='utf-8')
# Mapa cod_barri -> id_territorio (barris GUB)
gub_barri = dim_territorio[dim_territorio['nivel_territorial'] == 'barri'].copy()
gub_barri['cod_barri'] = gub_barri['cod_barri'].astype(int)
MAP_BARRI = gub_barri.set_index('cod_barri')['id_territorio'].to_dict()
print('Barris GUB en dim:', len(MAP_BARRI))

---
## 1. Renta a nivel barri (renta secció censal → barri)

Dos indicadores (2015-2021): renta tributaria neta y renta disponible de los hogares por persona.
Se agrega de sección censal a barri con la **media** de las secciones (aproximación: no hay
población por sección en estos ficheros para ponderar).

In [ ]:
def cargar_renta_barri(subdir, indicador):
    frames = []
    for f in sorted((OD / subdir).glob('*.csv')):
        df = pd.read_csv(f, encoding='utf-8-sig')
        frames.append(df)
    raw = pd.concat(frames, ignore_index=True)
    raw['Import_Euros'] = pd.to_numeric(raw['Import_Euros'], errors='coerce')
    raw = raw.dropna(subset=['Import_Euros'])
    # agregar secciones -> barri (media)
    agg = raw.groupby(['Any', 'Codi_Barri', 'Nom_Barri'], as_index=False)['Import_Euros'].mean()
    agg['indicador'] = indicador
    return agg

renta_trib = cargar_renta_barri('renda_tributaria', 'renta_tributaria_barri')
renta_disp = cargar_renta_barri('renda_disponible_llars', 'renta_disponible_barri')
renta = pd.concat([renta_trib, renta_disp], ignore_index=True)

renta['id_territori'] = renta['Codi_Barri'].map(MAP_BARRI).astype('Int64')
n_sin = renta['id_territori'].isna().sum()
print('Filas renta:', len(renta), '| sin id_territori (barri no en GUB):', n_sin)
print('Codi_Barri sin match:', sorted(renta[renta['id_territori'].isna()]['Codi_Barri'].unique()))

contexto_renta = pd.DataFrame({
    'anyo': renta['Any'].astype(int),
    'id_territori': renta['id_territori'],
    'territori': renta['Nom_Barri'],
    'nivel_territorial': 'barri',
    'indicador': renta['indicador'],
    'categoria': pd.NA,
    'sexe': pd.NA,
    'valor': renta['Import_Euros'].round().astype('Int64'),
    'unitat': 'euros'
})
print('\ncontexto_renta_barri:', contexto_renta.shape, '| años:', sorted(contexto_renta['anyo'].unique()))
print('indicadores:', contexto_renta['indicador'].unique().tolist())
contexto_renta.head(3)

In [ ]:
ruta_r = CLEAN / 'contexto_renta_barri.csv'
contexto_renta.to_csv(ruta_r, index=False, encoding='utf-8')
print(f'[OK] Guardado {ruta_r.name}: {len(contexto_renta)} filas')
# Cross-check: el Raval (cod_barri 1) renta tributaria 2018
raval = contexto_renta[(contexto_renta['territori']=='el Raval') & (contexto_renta['anyo']==2018) & (contexto_renta['indicador']=='renta_tributaria_barri')]
print('el Raval renta tributaria 2018 (media secciones):', raval['valor'].values)

---
## 2. Geometrías — barris (match por código)

In [ ]:
barris_geo = pd.read_csv(OD / 'opendataBarcelona_BarcelonaCiutat_Barris.csv', encoding='utf-8-sig')
barris_geo['id_territorio'] = barris_geo['codi_barri'].map(MAP_BARRI).astype('Int64')
n_sin_b = barris_geo['id_territorio'].isna().sum()
print('Barris con geometría:', len(barris_geo), '| sin id_territorio:', n_sin_b)

geo_barris = pd.DataFrame({
    'id_territorio': barris_geo['id_territorio'],
    'nivel_territorial': 'barri',
    'codi': barris_geo['codi_barri'],
    'nom': barris_geo['nom_barri'],
    'geometria_wgs84': barris_geo['geometria_wgs84']
}).dropna(subset=['id_territorio'])
print('geo_barris:', len(geo_barris))

---
## 3. Geometrías — ABP Mossos (match por nombre normalizado)

Los nombres difieren ('ABP Alt Empordà - Figueres' vs 'Alt Empordà-Figueres') → se normalizan
(quitar prefijo 'ABP ', unificar separadores) y se valida cuántos casan.

In [ ]:
import csv as _csv
_csv.field_size_limit(10**8)
abp_geo = pd.read_csv(OD / 'Áreas básicas policiales.csv', encoding='utf-8-sig', engine='python')
geom_col = [c for c in abp_geo.columns if 'GEOMETRIA' in c.upper() or 'the_geom' in c][0]
print('Cols ABP geo:', abp_geo.columns.tolist())

def norm_abp(s):
    s = str(s).strip()
    s = re.sub(r'^ABP\s+', '', s)            # quitar prefijo ABP
    s = s.lower()
    s = re.sub(r'\s*-\s*', '-', s)            # unificar ' - ' -> '-'
    s = re.sub(r'\s+', ' ', s).strip()
    return s

abp_dim = dim_territorio[dim_territorio['nivel_territorial'] == 'abp'][['id_territorio', 'abp']].copy()
abp_dim['key'] = abp_dim['abp'].map(norm_abp)
abp_geo['key'] = abp_geo['NOM_ABP_MOSSOS'].map(norm_abp)

merged = abp_dim.merge(abp_geo[['key', geom_col]], on='key', how='left')
n_match = merged[geom_col].notna().sum()
print(f'ABP en dim: {len(abp_dim)} | con geometría: {n_match} | sin: {len(abp_dim)-n_match}')
print('ABP sin geometría (muestra):', merged[merged[geom_col].isna()]['abp'].head(8).tolist())

geo_abp = pd.DataFrame({
    'id_territorio': merged['id_territorio'],
    'nivel_territorial': 'abp',
    'codi': pd.NA,
    'nom': merged['abp'],
    'geometria_wgs84': merged[geom_col]
}).dropna(subset=['geometria_wgs84'])
print('geo_abp:', len(geo_abp))

---
## 4. Unir geometrías y guardar

In [ ]:
geo = pd.concat([geo_barris, geo_abp], ignore_index=True)
geo['id_territorio'] = geo['id_territorio'].astype(int)
# Validación FK
fk_ok = geo['id_territorio'].isin(dim_territorio['id_territorio']).all()
print('geo_territorio:', geo.shape, '| FK id_territorio válido:', fk_ok)
print('Por nivel:', geo['nivel_territorial'].value_counts().to_dict())

ruta_g = CLEAN / 'geo_territorio.csv'
geo.to_csv(ruta_g, index=False, encoding='utf-8')
print(f'\n[OK] Guardado {ruta_g.name}: {len(geo)} filas')
print('\n[OK] nb08 completo: renta a nivel barri + geometrías incorporadas.')